In [1]:
import numpy as np
import pandas as pd

# **1. Create DataFrame**

In [2]:
data = {
    'Category': ['A', 'B', 'A', 'B', 'A', 'B', 'A', 'B'],
    'Store': ['S1', 'S1', 'S2', 'S2', 'S1', 'S2', 'S2', 'S1'],
    'Sales': [100, 200, 150, 250, 120, 180, 200, 300],
    'Quantity': [10, 15, 12, 18, 8, 20, 15, 25],
    'Date': pd.date_range('2023-01-01', periods=8)
}
df = pd.DataFrame(data)

In [3]:
df

,Category,Store,Sales,Quantity,Date
0,A,S1,100,10,2023-01-01
1,B,S1,200,15,2023-01-02
2,A,S2,150,12,2023-01-03
3,B,S2,250,18,2023-01-04
4,A,S1,120,8,2023-01-05
5,B,S2,180,20,2023-01-06
6,A,S2,200,15,2023-01-07
7,B,S1,300,25,2023-01-08


# **2. GroupBy**

**Group by Category**

In [4]:
cat = df.groupby('Category')['Sales'].sum()
cat

,Sales
Category,
A,570
B,930


In [5]:
cat = df.groupby('Category')['Sales'].sum().reset_index()
print(cat)

  Category  Sales
0        A    570
1        B    930


In [6]:
cat = df.groupby('Category', as_index=False)['Sales'].sum()
cat

,Category,Sales
0,A,570
1,B,930


**Group by Store**

In [7]:
cat = df.groupby('Store', as_index=False)['Sales'].sum()
cat

,Store,Sales
0,S1,720
1,S2,780


**Group by Multiple Columns**

In [8]:
cat = df.groupby(['Category','Store'], as_index=False)['Sales'].sum()
cat

,Category,Store,Sales
0,A,S1,220
1,A,S2,350
2,B,S1,500
3,B,S2,430


# **3. Aggregation**

**Single Aggregation**

In [9]:
df['Sales'].mean()

np.float64(187.5)

In [10]:
print(df['Sales'].mean())

187.5


**Multiple Aggregations**

In [11]:
df['Sales'].agg(['sum','mean','min','max','count','std','median'])

,Sales
sum,1500.000000
mean,187.500000
min,100.000000
max,300.000000
count,8.000000
std,66.062741
median,190.000000


# **4. GroupBy with Multiple Aggregations**

In [12]:
cat = df.groupby('Category')['Sales'].agg(['sum','mean','min','max','count']).reset_index()
cat

,Category,sum,mean,min,max,count
0,A,570,142.5,100,200,4
1,B,930,232.5,180,300,4


In [13]:
print(cat)

  Category  sum   mean  min  max  count
0        A  570  142.5  100  200      4
1        B  930  232.5  180  300      4


In [14]:
cat = df.groupby('Category').agg(
    Total=('Sales','sum'),
    Avg=('Sales','mean'),
    Min=('Sales','min'),                  # To Rename columns
    Max=('Sales','max'),
    Count=('Sales','count')
).reset_index()
cat

,Category,Total,Avg,Min,Max,Count
0,A,570,142.5,100,200,4
1,B,930,232.5,180,300,4


# **5. Different Aggregation for Different Columns**

In [15]:
cat = df.groupby('Category').agg({'Sales':'sum','Quantity':'mean'}).reset_index()
cat

,Category,Sales,Quantity
0,A,570,11.25
1,B,930,19.50


# **6. Named Aggregation**

In [16]:
cat = df.groupby('Category').agg(Total_Sales=('Sales','sum'),Avg_Qty=('Quantity','mean')).reset_index()
cat

,Category,Total_Sales,Avg_Qty
0,A,570,11.25
1,B,930,19.50


# **7. size() vs count()**

**size()**

In [17]:
df.groupby('Category').size().reset_index()     # Counts rows

,Category,0
0,A,4
1,B,4


**count()**

In [18]:
df.groupby('Category')['Sales'].count()    # Counts non-null values.

,Sales
Category,
A,4
B,4


# **8. reset_index()**

In [19]:
cat = df.groupby(['Category','Store'])['Sales'].sum().reset_index()
cat

,Category,Store,Sales
0,A,S1,220
1,A,S2,350
2,B,S1,500
3,B,S2,430


# **9. sort_values()**

In [20]:
cat = df.groupby('Category')['Sales'].sum().sort_values(ascending=False).reset_index()
cat

,Category,Sales
0,B,930
1,A,570


# **10. filter()**

In [21]:
cat = df.groupby('Category').filter(lambda x:x['Sales'].sum()>600)     # Keeps only Category B.
cat

,Category,Store,Sales,Quantity,Date
1,B,S1,200,15,2023-01-02
3,B,S2,250,18,2023-01-04
5,B,S2,180,20,2023-01-06
7,B,S1,300,25,2023-01-08


# **11. transform()**

In [22]:
df['Cat_Total'] = df.groupby('Category')['Sales'].transform('sum')
df['Cat_Total']

,Cat_Total
0,570
1,930
2,570
3,930
4,570
5,930
6,570
7,930


# **12) apply()**

In [29]:
res = df.groupby('Category')['Sales'].apply(      # When the operation is not a simple aggregation
    lambda x: (x.max() + x.min()) / 2             # No direct built-in function for this → apply() makes sense
).reset_index()
res

,Category,Sales
0,A,150.0
1,B,240.0


In [33]:
def custom_fn(x):
    val1 = x.max()
    val2 = x.min()
    diff = val1 - val2                      # Multiple-step logic inside groups

    if diff > 100:
        return diff
    else:
        return 0

res = df.groupby('Category')['Sales'].apply(custom_fn).reset_index()
res

,Category,Sales
0,A,0
1,B,120


**When you should NOT use apply()**

Simple aggregations

In [36]:
# ❌ Avoid
df.groupby('Category').apply(lambda x: x['Sales'].sum())

# ✅ Use
df.groupby('Category')['Sales'].sum()

/tmp/ipykernel_3777/460062841.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby('Category').apply(lambda x: x['Sales'].sum())


,Sales
Category,
A,570
B,930


**Can I do this using agg, transform, or direct column ops?**


*   YES → don’t use apply()
*   NO → use apply()

apply() is the last option, not the first.

| **Task**                 | **Use**               |
| -------------------- | ----------------- |
| sum, mean, max, min  | `agg()`           |
| column-wise ops      | direct operations |
| same shape output    | `transform()`     |
| complex/custom logic | `apply()`         |


# **13) GroupBy Multiple Columns Multiple Aggregates**

In [38]:
cat = df.groupby(['Category','Store']).agg({'Sales':['sum','mean'],'Quantity':'max'}).reset_index()    # hierarchical column index
cat

Category Store Sales        Quantity
                   sum   mean      max
0        A    S1   220  110.0       10
1        A    S2   350  175.0       15
2        B    S1   500  250.0       25
3        B    S2   430  215.0       20

In [40]:
grp = df.groupby(['Category', 'Store'])

res = grp.agg({
    'Sales': ['sum', 'mean'],
    'Quantity': 'max'
}).reset_index()

# flatten column names
res.columns = ['Category', 'Store', 'Sales_sum', 'Sales_mean', 'Quantity_max']   # flatten columns
res

,Category,Store,Sales_sum,Sales_mean,Quantity_max
0,A,S1,220,110.0,10
1,A,S2,350,175.0,15
2,B,S1,500,250.0,25
3,B,S2,430,215.0,20


# **14) GroupBy on Multiple Metrics**

In [42]:
cat = df.groupby('Category')[['Sales','Quantity']].sum().reset_index()
cat

,Category,Sales,Quantity
0,A,570,45
1,B,930,78


# **15) Cumulative Group Operations**

In [44]:
df['Running_Sales'] = df.groupby('Category')['Sales'].cumsum()
df['Running_Sales']

,Running_Sales
0,100
1,200
2,250
3,450
4,370
5,630
6,570
7,930


# **16) Date Grouping**

**Group by Month**

In [46]:
cat = df.groupby(df['Date'].dt.month)['Sales'].sum().reset_index()
cat

,Date,Sales
0,1,1500


**Group by Day Name**

In [47]:
cat = df.groupby(df['Date'].dt.day_name())['Sales'].sum().reset_index()
cat

,Date,Sales
0,Friday,180
1,Monday,200
2,Saturday,200
3,Sunday,400
4,Thursday,120
5,Tuesday,150
6,Wednesday,250


# **17) Pivot Table**

In [50]:
cat = pd.pivot_table(df, values='Sales', index='Category', columns='Store', aggfunc='sum').reset_index()
cat

Store,Category,S1,S2
0,A,220,350
1,B,500,430


# **18) as_index=False**

In [52]:
cat = df.groupby('Category',as_index=False)['Sales'].sum().reset_index()
cat

,index,Category,Sales
0,0,A,570
1,1,B,930


# **19) Important Aggregation Functions**

1.  sum()
2.  mean()
3.  median()
4.  min()
5.  max()
6.  count()
7.  size()
8.  std()
9.  var()
10. nunique()
11. first()
12. last()

# **20) Important GroupBy Parameters**


*   by=
*   as_index=
*   sort=
*   dropna=


# **Interview Important Topics**

1. groupby()
2. agg()
3. transform()
4. filter()
5. apply()
6. size vs count
7. reset_index()
8. pivot_table()
9. cumsum()
10. named aggregation

# **GroupBy Flow**

Split  -> groupby()

Apply  -> sum mean agg transform

Combine -> result